In [ ]:
# !pip install --upgrade pip

In [ ]:
!pip install evaluate
!pip install jiwer
!pip install datasets['audio']
!pip install transformers==4.26.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.3/100.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 kB 15.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 45.0 MB/s eta 0:00:00:00:01
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
ERROR: ERROR: Failed to build installable wh

In [ ]:
import torch
# import torchaudio
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from evaluate import load
from tqdm import tqdm
import torch
import re
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline

print("Imports successful!")
# print("Transformers version:", transformers.__version__)

Imports successful!


In [ ]:
# Cell 3: Configuration
DATASETS = {
    # "egyptian_asr": {
    #     "name": "MightyStudent/Egyptian-ASR-MGB-3",
    #     "split": "train", 
    #     "audio_col": "audio",
    #     "text_col": "sentence"
    # },
    "arabic_egy": {
        "name": "MAdel121/arabic-egy-cleaned", 
        "split": "test",
        "audio_col": "audio", 
        "text_col": "text"
    }
}

MODELS = {
    # "tiny": "openai/whisper-tiny",
    # "small": "openai/whisper-small",
    # "medium": "openai/whisper-medium",
    "large-v3": "openai/whisper-large"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [ ]:
def normalize_arabic_text(text):
    """
    Arabic text normalization:
    1. Remove punctuation
    2. Remove diacritics
    3. Eastern Arabic numerals to Western Arabic numerals

    Arguments
    ---------
    text: str
        text to normalize
    Output
    ---------
    normalized text
    """
    # Remove punctuation
    punctuation = r'[!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~،؛؟]'
    text = re.sub(punctuation, '', text)

    # Remove diacritics
    diacritics = r'[\u064B-\u0652]'  # Arabic diacritical marks (Fatha, Damma, etc.)
    text = re.sub(diacritics, '', text)
    
    # Normalize Hamzas and Maddas
    text = re.sub('پ', 'ب', text)
    text = re.sub('ڤ', 'ف', text)
    text = re.sub(r'[آ]', 'ا', text)
    text = re.sub(r'[أإ]', 'ا', text)
    text = re.sub(r'[ؤ]', 'و', text)
    text = re.sub(r'[ئ]', 'ي', text)
    text = re.sub(r'[ء]', '', text)   

    # Transliterate Eastern Arabic numerals to Western Arabic numerals
    eastern_to_western_numerals = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9'
    }
    for eastern, western in eastern_to_western_numerals.items():
        text = text.replace(eastern, western)

    return text.strip()


def evaluate_whisper(model_name, dataset_config, max_samples=20):
    """Evaluation function with normalization and simplified metrics"""
    
    print(f"\nTesting {model_name}...")
    
    try:
        # Load processor and model
        processor = WhisperProcessor.from_pretrained(model_name)
        model = WhisperForConditionalGeneration.from_pretrained(model_name)
        model = model.to(device)
        model.eval()
        
        # Load dataset
        dataset = load_dataset(dataset_config["name"], split=dataset_config["split"])
        
        # if len(dataset) > max_samples:
        #     dataset = dataset.select(range(max_samples))
        
        references = []
        predictions = []
        
        # Process each sample
        for sample in tqdm(dataset):
            audio = sample[dataset_config["audio_col"]]
            reference_text = sample[dataset_config["text_col"]]
            
            # Get audio array and sampling rate
            audio_array = audio["array"]
            sampling_rate = audio["sampling_rate"]
            
            # Process audio for Whisper
            inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt").to(device)
            
            # Generate transcription with language forcing
            forced_decoder_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")
            
            with torch.no_grad():
                generated_ids = model.generate(**inputs, forced_decoder_ids=forced_decoder_ids)
            
            transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
            
            # Apply normalization to both reference and prediction
            references.append(normalize_arabic_text(reference_text))
            predictions.append(normalize_arabic_text(transcription))
        
        # # Calculate metrics using evaluate library
        # wer_metric = load("wer")
        # cer_metric = load("cer")
        
        # wer_score = wer_metric.compute(references=references, predictions=predictions)
        # cer_score = cer_metric.compute(references=references, predictions=predictions)


        wer_score = wer(references, predictions)
        cer_score = cer(references, predictions)
        
        exact_matches = sum(1 for ref, pred in zip(references, predictions) if ref.strip() == pred.strip())
        accuracy = exact_matches / len(references)
        
        return {
            "model": model_name,
            "dataset": dataset_config["name"],
            "wer": wer_score,
            "cer": cer_score,
            "accuracy": accuracy,
            "samples_tested": len(references),
            "references": references,
            "predictions": predictions
        }
    
    except Exception as e:
        print(f"Error with {model_name}: {str(e)[:200]}...")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
# Cell 5: Run evaluation
all_results = []

for dataset_name, dataset_config in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*60}")
    
    for model_name, model_path in MODELS.items():
        result = evaluate_whisper(model_path, dataset_config, max_samples=50)
        
        if result is not None:
            all_results.append(result)
            print(f"{model_name}: WER={result['wer']:.3f}, CER={result['cer']:.3f}")

NameError: name 'DATASETS' is not defined